<a href="https://colab.research.google.com/github/frctlprdx/Face-Detection/blob/main/TraingSVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install facenet-pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 87.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.

In [ ]:
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from facenet_pytorch import MTCNN, InceptionResnetV1
from PIL import Image
import numpy as np
import os
import tensorflow as tf
import shutil

In [ ]:
# Inisialisasi MTCNN dan model
mtcnn = MTCNN()
resnet = InceptionResnetV1(pretrained='casia-webface').eval()

# Folder data train
train_dir = '/content/drive/MyDrive/Joined/augmented'  # Ganti dengan path ke folder data augmented
embedding_dict = {}  # Dictionary untuk menyimpan embeddings berdasarkan label

# Loop melalui setiap subfolder dalam folder train
for class_name in os.listdir(train_dir):
    class_dir = os.path.join(train_dir, class_name)
    if os.path.isdir(class_dir):
        for img_name in os.listdir(class_dir):
            img_path = os.path.join(class_dir, img_name)
            img = Image.open(img_path)

            # Ekstrak embeddings
            aligned = mtcnn(img)
            if aligned is not None:  # Pastikan wajah terdeteksi
                aligned = aligned.unsqueeze(0)  # Tambahkan dimensi batch
                embedding = resnet(aligned).detach().numpy()

                # Simpan embedding ke dalam dictionary berdasarkan label
                if class_name not in embedding_dict:
                    embedding_dict[class_name] = []  # Inisialisasi list untuk label baru
                embedding_dict[class_name].append(embedding)  # Tambahkan embedding ke list

# Menyimpan dictionary ke dalam file
np.save('/content/drive/MyDrive/Colab Notebooks/train_embeddings_with_labels.npy', embedding_dict)


  0%|          | 0.00/111M [00:00<?, ?B/s]

In [ ]:
# Inisialisasi MTCNN dan model
mtcnn = MTCNN()
resnet = InceptionResnetV1(pretrained='casia-webface').eval()

# Folder data train
val_dir = '/content/drive/MyDrive/normalized/val'  # Ganti dengan path ke folder data augmented
embedding_dict = {}  # Dictionary untuk menyimpan embeddings berdasarkan label

# Loop melalui setiap subfolder dalam folder train
for class_name in os.listdir(val_dir):
    class_dir = os.path.join(val_dir, class_name)
    if os.path.isdir(class_dir):
        for img_name in os.listdir(class_dir):
            img_path = os.path.join(class_dir, img_name)
            img = Image.open(img_path)

            # Ekstrak embeddings
            aligned = mtcnn(img)
            if aligned is not None:  # Pastikan wajah terdeteksi
                aligned = aligned.unsqueeze(0)  # Tambahkan dimensi batch
                embedding = resnet(aligned).detach().numpy()

                # Simpan embedding ke dalam dictionary berdasarkan label
                if class_name not in embedding_dict:
                    embedding_dict[class_name] = []  # Inisialisasi list untuk label baru
                embedding_dict[class_name].append(embedding)  # Tambahkan embedding ke list

# Menyimpan dictionary ke dalam file
np.save('/content/drive/MyDrive/Colab Notebooks/val_embeddings_with_labels.npy', embedding_dict)


In [ ]:
import numpy as np
file_path = '/content/drive/MyDrive/Colab Notebooks/val_embeddings_with_labels.npy'

# Membaca file .npy
data = np.load(file_path, allow_pickle=True)

data

array({'S001': [array([[ 0.03584767, -0.04589787, -0.02501755, -0.0238809 , -0.08440839,
        -0.07223236, -0.01109333,  0.02344843, -0.05873272, -0.00083214,
        -0.03022857,  0.00996567,  0.0106579 ,  0.02849039, -0.0107803 ,
         0.09939578, -0.0625075 , -0.02706509,  0.00132298,  0.01455669,
        -0.06798172,  0.05358215,  0.00231709,  0.04493013, -0.00547315,
         0.03946921, -0.05086716, -0.04267091,  0.00042737,  0.02240677,
        -0.09424873, -0.02325678, -0.05365201, -0.00609044, -0.0031404 ,
         0.00542845, -0.06419594, -0.07550848, -0.05726942,  0.00689382,
        -0.0305875 , -0.02854141, -0.0642624 ,  0.13189991,  0.04418096,
         0.04399946,  0.06215249, -0.0837232 ,  0.06644715, -0.0197427 ,
         0.01163494,  0.04203943,  0.0034826 ,  0.0482862 , -0.05727068,
         0.02301378,  0.00512465, -0.07197987, -0.05527034,  0.00885115,
         0.01097812, -0.03993614,  0.03581868, -0.06028974, -0.01245123,
         0.03300795,  0.07898755, -

In [ ]:
from sklearn import svm
from sklearn.metrics import accuracy_score

In [ ]:
# Load embeddings dari file
train_embeddings = np.load('/content/drive/MyDrive/Colab Notebooks/train_embeddings_with_labels.npy', allow_pickle=True).item()
val_embeddings = np.load('/content/drive/MyDrive/Colab Notebooks/val_embeddings_with_labels.npy', allow_pickle=True).item()

# Mengumpulkan data train
train_X = []
train_y = []

for label, embeddings in train_embeddings.items():
    for embedding in embeddings:
        if np.ndim(embedding) == 0:
            embedding = embedding.item()
            embedding = np.array([embedding])  # Ubah menjadi array 1D
        train_X.append(embedding.flatten())  # Rata-ratakan ke 1D
        train_y.append(label)

# Mengumpulkan data val
val_X = []
val_y = []

for label, embeddings in val_embeddings.items():
    for embedding in embeddings:
        if np.ndim(embedding) == 0:
            embedding = embedding.item()
            embedding = np.array([embedding])  # Ubah menjadi array 1D
        val_X.append(embedding.flatten())  # Rata-ratakan ke 1D
        val_y.append(label)

# Konversi ke array NumPy
train_X = np.array(train_X)
train_y = np.array(train_y)
val_X = np.array(val_X)
val_y = np.array(val_y)

# Inisialisasi dan latih model SVM
clf = svm.SVC(kernel='linear')
clf.fit(train_X, train_y)

# Prediksi pada data validation
val_predictions = clf.predict(val_X)

# Hitung akurasi
accuracy = accuracy_score(val_y, val_predictions)
print(f"Akurasi model: {accuracy * 100:.2f}%")


Akurasi model: 92.90%


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# Prediksi pada data validation
val_predictions = clf.predict(val_X)

# Hitung akurasi
accuracy = accuracy_score(val_y, val_predictions)
print(f"Akurasi model: {accuracy * 100:.2f}%")

# Menampilkan laporan per klasifikasi untuk perbandingan hasil prediksi dengan label sebenarnya
print("\nClassification Report:")
print(classification_report(val_y, val_predictions))

# Menyimpan dan menampilkan data yang salah dikenali
misclassified_indices = np.where(val_predictions != val_y)[0]
print("\nData yang salah dikenali:")
for index in misclassified_indices:
    print(f"Index: {index}, Label Asli: {val_y[index]}, Prediksi: {val_predictions[index]}")


Akurasi model: 92.90%

Classification Report:
              precision    recall  f1-score   support

        S001       1.00      1.00      1.00         3
        S006       1.00      1.00      1.00         1
        S008       0.50      1.00      0.67         1
        S013       1.00      1.00      1.00         1
        S015       1.00      1.00      1.00         1
        S022       0.50      1.00      0.67         1
        S023       1.00      1.00      1.00         2
        S027       1.00      1.00      1.00         2
        S029       1.00      1.00      1.00         2
        S030       1.00      1.00      1.00         2
        S031       1.00      1.00      1.00         2
        S032       1.00      1.00      1.00         1
        S033       0.50      1.00      0.67         1
        S036       1.00      1.00      1.00         1
        S044       1.00      1.00      1.00         2
        S045       1.00      1.00      1.00         1
        S046       1.00      1.00  

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
